In [ ]:
%pip install mstarpy
import numpy as np
import datetime
import pandas as pd
import mstarpy
import time
from tqdm import tqdm

# Define list of funds to analyze
funds = [

    # Large-cap Value funds

    # "TRVLX",   
    # "VIVAX",         
    # "DODGX",  
    # "PEYAX",       
    # "OAKMX",      
    # "LBSAX",        
    # "FEQIX",
    # "SINAX",
    # "OIEIX",
    # "ALVIX",


  #-- NEW 5
    #  "FVALX",
    #  "VVIAX",
    #  "ACSTX",
    #  "MVIAX",
    #  "TILVX"


    # Large-cap Balanced funds            
 

    # "ABALX",
    # "JABAX",
    # "CAIBX",
    # "VWELX",
    # "CBLAX",
    # "MSFRX",
    # "FBALX",
    # "VBAIX",
    # "DODBX",
    # "SWOBX"


  #-- NEW 5
    # "FLCPX",
    # "FPURX",
    # "PRWCX",
    # "MAPOX",
    # "PABAX"



    


    # Large-Cap Income Funds

    # "PRFDX",   
    # "TWEIX",
    # "AMECX",
    # "LEIFX",
    # "FEQIX"
    # "VEIPX",
    # "SWDSX",
    # "MCDVX",
    # "WDIAX",
    # "AUIAX"


  #-- NEW 5
    # "VHYAX",
    # "FSDIX",
    # "VDADX",
    # "MADVX",
    # "CDGIX"
    
 



      # Large-Cap Growth Funds

        # "AMAGX",
        # "DREVX",
        # "FCNTX",
        # "APGAX",
        # "MIGFX",
        # "CFGRX",
        # "MFEIX",
        # "CMLIX",
        # "LEGAX",
        # "TRBCX"


  #-- NEW 5
# "TILIX",
# "LGPIX",
# "VIGAX",
# "RYOCX",
# "FSPGX"          
]


# Define date range for the analysis
start_date = datetime.datetime(2003, 1, 1)
end_date = datetime.datetime(2025, 6, 30)

# Risk-free rate
risk_free_rate = 0.00

def calculate_mdd_for_period(group):
    cumulative_returns = (group['daily_return'] + 1).cumprod()
    cumulative_returns = cumulative_returns.replace([np.inf, -np.inf], np.nan).fillna(1)
    rolling_max = cumulative_returns.cummax()
    drawdowns = (cumulative_returns - rolling_max) / rolling_max
    mdd = drawdowns.min()
    return mdd

def calculate_semi_metrics(group):
    semi_return = (group['daily_return'] + 1).prod() - 1
    if np.isinf(semi_return) or np.isnan(semi_return):
        semi_return = -1
    volatility = group['daily_return'].std() * np.sqrt(252)  # Annualized
    sharpe_ratio = (semi_return - risk_free_rate) / volatility if volatility != 0 else np.nan
    mdd = calculate_mdd_for_period(group)
    return pd.Series({
        'Semi_Annual_Return': semi_return,
        'Volatility': volatility,
        'Sharpe_Ratio': sharpe_ratio,
        'MDD': mdd
    })

master_df = pd.DataFrame()

print("Fetching and processing fund data...")
for fund_ticker in tqdm(funds):
    try:
        print(f"\nProcessing {fund_ticker}")
        fund = mstarpy.Funds(term=fund_ticker, country="us")
        print(f"Fetching historical data for {fund_ticker}...")
        history = fund.nav(start_date=start_date, end_date=end_date, frequency="daily")
        df = pd.DataFrame(history)

        if df.empty:
            print(f"No data retrieved for {fund_ticker}. Skipping.")
            time.sleep(3)
            continue

        df['date'] = pd.to_datetime(df['date'])
        df = df.sort_values(by='date')
        df['daily_return'] = df['nav'].pct_change()
        df = df.dropna(subset=['daily_return'])

        # Add semi-annual period column
        df['Semi'] = df['date'].dt.month.apply(lambda x: 1 if x <= 6 else 2)
        df['Year_Semi'] = df['date'].dt.year.astype(str) + "-H" + df['Semi'].astype(str)
        df = df.fillna(0)

        semi_metrics = df.groupby('Year_Semi').apply(calculate_semi_metrics).reset_index()
        semi_metrics['Fund'] = fund_ticker

        master_df = pd.concat([master_df, semi_metrics[['Year_Semi', 'Fund', 'Semi_Annual_Return', 'Sharpe_Ratio']]])

        print(f"Successfully processed {fund_ticker}")
        time.sleep(5)

    except Exception as e:
        print(f"Error processing {fund_ticker}: {str(e)}")
        time.sleep(3)
        continue

# Calculate ranks for each semi-annual period
print("Calculating performance rankings...")
final_df = pd.DataFrame()

for period in master_df['Year_Semi'].unique():
    period_data = master_df[master_df['Year_Semi'] == period].copy()
    period_data['Return_Rank'] = period_data['Semi_Annual_Return'].rank(ascending=False).astype(int)
    period_data['Sharpe_Rank'] = period_data['Sharpe_Ratio'].rank(ascending=False).astype(int)
    final_df = pd.concat([final_df, period_data])

# Format output
final_df['Semi_Annual_Return'] = final_df['Semi_Annual_Return'].apply(lambda x: f"{x:.2%}")
final_df['Sharpe_Ratio'] = final_df['Sharpe_Ratio'].apply(lambda x: f"{x:.2f}" if not pd.isna(x) else "N/A")

final_df = final_df[['Year_Semi', 'Fund', 'Semi_Annual_Return', 'Sharpe_Ratio', 'Return_Rank', 'Sharpe_Rank']]

# Sort chronologically
final_df['SortKey'] = final_df['Year_Semi'].str.extract(r'(\d+)-H(\d)')\
                .apply(lambda x: (int(x[0]), int(x[1])), axis=1)
final_df = final_df.sort_values(by=['SortKey', 'Fund']).drop(columns='SortKey')

print("\nFinal Semi-Annual Performance Table:")
print(final_df)

# Save results
final_df.to_csv("mutual_fund_semi_annual_growth2.csv", index=False)
print("\nResults saved to 'mutual_fund_semi_annual_growth2.csv'")


In [ ]:
import pandas as pd
df1=pd.read_csv("mutual_fund_semi_annual_income1.csv")
df2=pd.read_csv("mutual_fund_semi_annual_income2.csv")
df=pd.concat([df1, df2], ignore_index=True)
df=df.drop(columns=['Return_Rank','Sharpe_Rank'], axis=1)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Function to convert string percentages to float
def convert_percentage(x):
    if isinstance(x, str) and '%' in x:
        return float(x.strip('%')) / 100
    return float(x)

# === START: Prepare Data ===
df_calc = df.copy()
df_calc['Semi_Annual_Return_Float'] = df_calc['Semi_Annual_Return'].apply(convert_percentage)
df_calc['Sharpe_Ratio'] = df_calc['Sharpe_Ratio'].astype(float)

df_extended = df_calc.copy()
periods = sorted(df_extended['Year_Semi'].unique())
funds = sorted(df_extended['Fund'].unique())

# Rank funds each period
df_extended = df_extended.assign(
    Return_Rank=df_extended.groupby('Year_Semi')['Semi_Annual_Return_Float']
                         .rank(ascending=False, method='first'),
    Sharpe_Rank=df_extended.groupby('Year_Semi')['Sharpe_Ratio']
                         .rank(ascending=False, method='first')
)

# === Strategy Implementation Function ===
def implement_strategy(data, start_amount=100, rank_column='Return_Rank', target_rank=1):
    prs = sorted(data['Year_Semi'].unique())
    perf = []
    amt = start_amount
    label = f"{target_rank}{'st' if target_rank==1 else 'nd' if target_rank==2 else 'rd' if target_rank==3 else 'th'} Best by {'Returns' if rank_column=='Return_Rank' else 'Sharpe'}"

    first = prs[0]
    r0 = data.loc[data['Year_Semi']==first, 'Semi_Annual_Return_Float'].mean()
    amt *= (1 + r0)
    perf.append({'Year_Semi': first, 'Strategy': label, 'Amount': amt,
                 'Return': r0, 'Volatility': data.loc[data['Year_Semi']==first, 'Semi_Annual_Return_Float'].std()})

    for i in range(1, len(prs)):
        pr = prs[i]
        prev = prs[i-1]
        ranked = data.loc[data['Year_Semi']==prev].sort_values(rank_column)
        if len(ranked) >= target_rank:
            f = ranked.iloc[target_rank-1]['Fund']
            row = data[(data['Year_Semi']==pr) & (data['Fund']==f)]
            if not row.empty:
                ret = row['Semi_Annual_Return_Float'].iloc[0]
                amt *= (1 + ret)
            else:
                ret = 0
        else:
            ret = 0
        vol = data.loc[data['Year_Semi']==pr, 'Semi_Annual_Return_Float'].std()
        perf.append({'Year_Semi': pr, 'Strategy': label, 'Amount': amt,
                     'Return': ret, 'Volatility': vol})
    return pd.DataFrame(perf)

# === Run Strategies ===

# Return-based
return_strategies = [implement_strategy(df_extended, 100, 'Return_Rank', r) for r in range(1, 11)]
return_strategy_df = pd.concat(return_strategies)

# Sharpe-based
sharpe_strategies = [implement_strategy(df_extended, 100, 'Sharpe_Rank', r) for r in range(1, 11)]
sharpe_strategy_df = pd.concat(sharpe_strategies)

# Individual Funds
fund_perf = []
for fund in funds:
    df_f = df_extended[df_extended['Fund']==fund]
    val = 100
    for pr in periods:
        row = df_f[df_f['Year_Semi']==pr]
        if not row.empty:
            r = row['Semi_Annual_Return_Float'].iloc[0]
            val *= (1 + r)
            vol = df_extended.loc[df_extended['Year_Semi']==pr, 'Semi_Annual_Return_Float'].std()
            fund_perf.append({'Year_Semi': pr, 'Fund': fund, 'Amount': val,
                              'Return': r, 'Volatility': vol})
fund_performance_df = pd.DataFrame(fund_perf)

# === Summary Metrics ===
fund_avg_returns = fund_performance_df.groupby('Fund')['Return'].mean()
fund_final_values = {
    fund: fund_performance_df[(fund_performance_df['Fund']==fund) & (fund_performance_df['Year_Semi']==periods[-1])]['Amount'].values[0]
    for fund in funds if not fund_performance_df[(fund_performance_df['Fund']==fund) & (fund_performance_df['Year_Semi']==periods[-1])].empty
}
return_strategy_avg_returns = return_strategy_df.groupby('Strategy')['Return'].mean()
sharpe_strategy_avg_returns = sharpe_strategy_df.groupby('Strategy')['Return'].mean()
fund_avg_sharpe = df_extended.groupby('Fund')['Sharpe_Ratio'].mean()

plt.style.use('seaborn-v0_8-darkgrid')

# === Plots ===

# 📈 1. Fund Performance
plt.figure(figsize=(14, 8))
for fund in funds:
    fp = fund_performance_df[fund_performance_df['Fund']==fund]
    avg_r = fund_avg_returns[fund]
    fv = fund_final_values.get(fund, 0)
    lbl = f"{fund} (Avg: {avg_r*100:.2f}%, Final: ${fv:.2f})"
    plt.plot(fp['Year_Semi'], fp['Amount'], label=lbl, linewidth=2)
plt.title('Fund Performance Over Semi-Annual Periods (Initial $100)')
plt.xlabel('Semi-Annual Period')
plt.ylabel('Portfolio Value ($)')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.legend(fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# 📈 2. Return-Based Strategies
plt.figure(figsize=(14, 8))
for strat in return_strategy_df['Strategy'].unique():
    sd = return_strategy_df[return_strategy_df['Strategy']==strat]
    avg_r = return_strategy_avg_returns[strat]
    fv = sd[sd['Year_Semi']==periods[-1]]['Amount'].iloc[0]
    lbl = f"{strat} (Avg: {avg_r*100:.2f}%, Final: ${fv:.2f})"
    plt.plot(sd['Year_Semi'], sd['Amount'], label=lbl, linewidth=2)
plt.title('Return-Based Strategy Performance (Semi-Annual)')
plt.xlabel('Semi-Annual Period')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.ylabel('Value ($)')
plt.legend(fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# 📈 3. Sharpe-Based Strategies
plt.figure(figsize=(14, 8))
for strat in sharpe_strategy_df['Strategy'].unique():
    sd = sharpe_strategy_df[sharpe_strategy_df['Strategy']==strat]
    avg_r = sharpe_strategy_avg_returns[strat]
    fv = sd[sd['Year_Semi']==periods[-1]]['Amount'].iloc[0]
    lbl = f"{strat} (Avg: {avg_r*100:.2f}%, Final: ${fv:.2f})"
    plt.plot(sd['Year_Semi'], sd['Amount'], label=lbl, linewidth=2)
plt.title('Sharpe-Based Strategy Performance (Semi-Annual)')
plt.xlabel('Semi-Annual Period')
plt.ylabel('Value ($)')
plt.legend(fontsize=9)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# 📈 4. Volatility Over Time
plt.figure(figsize=(12, 6))
vol_df = df_extended.groupby('Year_Semi')['Semi_Annual_Return_Float'].std().reset_index()
sns.barplot(x='Year_Semi', y='Semi_Annual_Return_Float', data=vol_df, palette='coolwarm')
plt.title('Volatility of Returns Over Semi-Annual Periods')
plt.xlabel('Semi-Annual Period')
plt.ylabel('Volatility (Std Dev of Returns)')
plt.xticks(rotation=45)
plt.tight_layout(); plt.show()

# === Final Summary Table ===

# 1) Select first 10 funds
funds_to_show = funds[:10]

# 2) Pivot Fund Performance
fund_pivot = (
    fund_performance_df
      .pivot(index='Year_Semi', columns='Fund', values='Amount')
      .loc[:, funds_to_show]
      .reset_index()
)

# 3) Pivot Return-based Strategies
return_pivot = (
    return_strategy_df
      .pivot(index='Year_Semi', columns='Strategy', values='Amount')
      .filter(like='Best by Returns')
      .reset_index()
)

# 4) Pivot Sharpe-based Strategies
sharpe_pivot = (
    sharpe_strategy_df
      .pivot(index='Year_Semi', columns='Strategy', values='Amount')
      .filter(like='Best by Sharpe')
      .reset_index()
)

# 5) Merge all
final_table = fund_pivot.merge(return_pivot, on='Year_Semi', how='left')
final_table = final_table.merge(sharpe_pivot, on='Year_Semi', how='left')

# 6) Rename Strategy Columns
for c in final_table.columns:
    if 'Best by Returns' in c:
        idx = int(c.split()[0][0])
        suffix = 'st' if idx==1 else 'nd' if idx==2 else 'rd' if idx==3 else 'th'
        final_table.rename(columns={c: f"{idx}{suffix} Best Return Strategy"}, inplace=True)
    elif 'Best by Sharpe' in c:
        idx = int(c.split()[0][0])
        suffix = 'st' if idx==1 else 'nd' if idx==2 else 'rd' if idx==3 else 'th'
        final_table.rename(columns={c: f"{idx}{suffix} Best Sharpe Strategy"}, inplace=True)

# Show Final Table
print(final_table)
